## PMOD-ACL2 Accelerometer (SPI Sensor)

This notebook shows how to use the Digilent PMOD-ACL2 3-axis accelerometer with the PYNQ-Z2 board through the AXI Quad SPI interface.
The accelerometer is based on the Analog Devices ADXL362 chip and communicates using SPI Mode 0.

The notebook initializes the SPI controller, verifies the device ID, enables measurement mode, and reads acceleration data from the X, Y, and Z axes.

References

ADXL362 Datasheet – https://www.analog.com/media/en/technical-documentation/data-sheets/ADXL362.pdf

Digilent PMOD-ACL2 Reference Manual – https://digilent.com/reference/pmod/pmodacl2/start

Xilinx AXI Quad SPI PG153 Documentation – https://docs.xilinx.com/r/en-US/pg153-axi-quad-spi

In [6]:
from pynq import Overlay, MMIO
import time

# ======================================================
# 1️⃣ Load Overlay
# ======================================================
ol = Overlay("car.bit", ignore_version=True)
print("Configuring SPI controller...")

# ======================================================
# 2️⃣ Select SPI IP from Overlay
# ======================================================
SPI_NAME = 'axi_quad_spi_0'   # adjust if different
SPI_BASE = ol.ip_dict[SPI_NAME]['phys_addr']
SPI_RANGE = ol.ip_dict[SPI_NAME]['addr_range']

spi = MMIO(SPI_BASE, SPI_RANGE)
print(f"✅ MMIO ready for {SPI_NAME} at {hex(SPI_BASE)}")

# ======================================================
# 3️⃣ SPI Register Offsets (from PG153)
# ======================================================
SRR    = 0x40   # Software Reset
SPICR  = 0x60   # Control Register
SPISR  = 0x64   # Status Register
SPIDTR = 0x68   # Transmit Register
SPIDRR = 0x6C   # Receive Register
SPISSR = 0x70   # Slave Select Register

# ======================================================
# 4️⃣ Initialize SPI in Master Mode
# ======================================================
spi.write(SRR, 0x0000000A)
time.sleep(0.05)

SPICR_CONFIG = (1 << 2) | (1 << 1) | (1 << 5) | (1 << 6)   # Master + Enable + FIFO resets
spi.write(SPICR, SPICR_CONFIG)
spi.write(SPISSR, 0xFFFF)  # deselect all slaves

print("✅ SPI controller initialized and ready for sensor use.")


Configuring SPI controller...
✅ MMIO ready for axi_quad_spi_0 at 0x41e20000
✅ SPI controller initialized and ready for sensor use.


In [7]:
# ======================================================
# PMOD-ACL2 (ADXL362 Accelerometer) SPI Interface
# ======================================================

READ_REG  = 0x0B
WRITE_REG = 0x0A

def spi_transfer_byte(val):
    spi.write(SPIDTR, val & 0xFF)
    time.sleep(0.0005)
    return spi.read(SPIDRR) & 0xFF

def spi_select(slave=0):
    spi.write(SPISSR, 0xFFFF & ~(1 << slave))

def spi_deselect():
    spi.write(SPISSR, 0xFFFF)

def adxl362_write_reg(addr, value):
    spi_select()
    spi_transfer_byte(WRITE_REG)
    spi_transfer_byte(addr)
    spi_transfer_byte(value)
    spi_deselect()

def adxl362_read_reg(addr):
    spi_select()
    spi_transfer_byte(READ_REG)
    spi_transfer_byte(addr)
    val = spi_transfer_byte(0x00)
    spi_deselect()
    return val

def adxl362_read_xyz():
    spi_select()
    spi_transfer_byte(READ_REG)
    spi_transfer_byte(0x0E)   # XDATA_L
    xL = spi_transfer_byte(0)
    xH = spi_transfer_byte(0)
    yL = spi_transfer_byte(0)
    yH = spi_transfer_byte(0)
    zL = spi_transfer_byte(0)
    zH = spi_transfer_byte(0)
    spi_deselect()

    def to_signed(lo, hi):
        val = (hi << 8) | lo
        return val - 65536 if val & 0x8000 else val

    return to_signed(xL, xH), to_signed(yL, yH), to_signed(zL, zH)

# --- Verify device ID ---
dev_id = adxl362_read_reg(0x00)
print(f"✅ Device ID (should be 0xAD): 0x{dev_id:02X}")

# --- Enable measurement mode ---
adxl362_write_reg(0x2D, 0x02)
time.sleep(0.05)
print("✅ Sensor in measurement mode.")

# --- Read acceleration ---
x, y, z = adxl362_read_xyz()
print(f"✅ X:{x}  Y:{y}  Z:{z}")


✅ Device ID (should be 0xAD): 0xFF
✅ Sensor in measurement mode.
✅ X:-1  Y:-1  Z:-1


In [3]:
import time
print("🔄 Streaming accelerometer data...")
while True:
    x, y, z = adxl362_read_xyz()
    print(f"X:{x:6d}  Y:{y:6d}  Z:{z:6d}")
    time.sleep(0.5)


🔄 Streaming accelerometer data...
X:    -1  Y:    -1  Z:    -1
X:    -1  Y:    -1  Z:    -1
X:    -1  Y:    -1  Z:    -1
X:    -1  Y:    -1  Z:    -1
X:    -1  Y:    -1  Z:    -1
X:    -1  Y:    -1  Z:    -1


KeyboardInterrupt: 